# Phase 12 — Geometric Lens (portiert) + Prämissen-Check

**A** Prämisse: überlebt der Cell-27A-Kohärenzvorsprung den Abzug der globalen
Router-Richtung? (reine Gewichtsanalyse, Sekunden)

**B** Geometric Lens auf unser Modell portiert: Ziel-Token ab `start_layer`
blind für seinen Kontext, Sweep über die 10 Voll-Attention-Layer, Druck-Readout.

**C** Kipp-Raten an vier Startlayern als grobe Bestätigung.

Selbstversorgend — **frische Runtime**, dann nur diese Zelle. ~15 min.
Braucht `vocab_foreign_masks.npz` und `banned_experts.json` auf Drive.

In [ ]:
# === Geometric Lens (portiert) + Praemissen-Check ==========================
# TEIL A: Praemisse von Cell 27A. Die absolute Router-Kohaerenz liegt bei 0.52
#   - auch bei der Kontrollgruppe. Verdacht: globale Vorzugsrichtung der
#   Router-Matrix. Test: globale Richtung aus allen Zeilen abziehen, dann
#   banned vs. Kontrolle erneut vergleichen. Reine Gewichtsanalyse, Sekunden.
# TEIL B: Portierung der Geometric Lens (Ma & Wolfinger, arXiv:2607.10578,
#   Repo Erikiss/geometric-lens, _make_attn_hook): Token t ist AB start_layer
#   blind fuer seinen Kontext (nur Selbst-Attention). Unser Cell-29-Eingriff,
#   aber layer-aufgeloest. Sweep ueber die 10 Voll-Attention-Layer der Hybrid-
#   Architektur, fuer Ziel-Token 43 (' name'), Attribut 42 (' local') und ein
#   Kontroll-Token. Readout: unterdrueckte Fremdschrift-Masse an den ersten
#   Antwort-Positionen (Druck wie Cell 17) - fein aufgeloest und billig.
# TEIL C: grobe Bestaetigung ueber die Kipp-Rate an vier Startlayern.
# Ergibt die TIEFENAUFLOESUNG des Ohrs, die wir bisher nicht hatten.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
# ---------------- Selbstversorgung: Modell + Prompts sicherstellen ----------
import glob, json, gc
for _n in ("model_b","tok_b"):                  # Base-Reste aus Cell 28 raus
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig): vermutlich "
        "belegt noch ein frueheres Modell den Speicher. Loesung: Laufzeit -> "
        "Sitzung neu starten, dann NUR diese Zelle ausfuehren.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN",'geometric_lens')
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
N_ANS=32; MAX_NEW=24; CHUNK=8; TOP_K=8; SEED=0
BANNED_JSON=globals().get("BANNED_JSON",
    (glob.glob("/content/drive/MyDrive/**/banned_experts.json",recursive=True) or [""])[0])
MASK_NPZ=(glob.glob("/content/drive/MyDrive/**/vocab_foreign_masks.npz",recursive=True) or [""])[0]
SCAFF="<|im_start|>user\n"
def think_prefix(u,th=""):
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
# ---------------- pure Logik (testbar) --------------------------------------
def blind_row(M,t,neg):
    """Zeile t der 4D-Maske: alles zu ausser Selbst-Attention (Kopie)"""
    M2=M.clone()
    if M2.dtype==torch.bool:
        M2[:,:,t,:]=False; M2[:,:,t,t]=True
    else:
        M2[:,:,t,:]=neg;   M2[:,:,t,t]=0
    return M2
def active_layers(all_layers,start):
    """Voll-Attention-Layer, ab denen der Eingriff greift"""
    return [l for l in all_layers if start is not None and l>=start]
def strip_global(Gn,eps=1e-9):
    """globale mittlere Router-Richtung aus normierten Zeilen entfernen"""
    mu=Gn.mean(0); mu=mu/mu.norm().clamp_min(eps)
    P=Gn-(Gn@mu).unsqueeze(1)*mu.unsqueeze(0)
    return P/P.norm(dim=1,keepdim=True).clamp_min(eps),mu
def coh(R,eps=1e-9):
    Rn=R/R.norm(dim=1,keepdim=True).clamp_min(eps)
    M=(Rn@Rn.T).abs(); iu=torch.triu_indices(R.shape[0],R.shape[0],offset=1)
    return float(M[iu[0],iu[1]].median())
def sign_p(nplus,n):
    if n==0: return 1.0
    t=min(nplus,n-nplus)
    return min(1.0,2*sum(math.comb(n,i) for i in range(0,t+1))/2**n)
def twoprop(k1,n1,k2,n2):
    p=(k1+k2)/(n1+n2); se=math.sqrt(p*(1-p)*(1/n1+1/n2)) if 0<p<1 else 0.0
    if se==0: return 1.0
    z=abs(k1/n1-k2/n2)/se
    return 2*(1-0.5*(1+math.erf(z/math.sqrt(2))))
def transition(profile,base,frac=0.5):
    """kleinster start_layer, ab dem der Druck unter frac*base faellt
       (profile: [(start_layer, wert)], aufsteigend nach layer)"""
    for l,v in profile:
        if v<frac*base: return l
    return None
def tok_span(offs,c0,c1):
    return [i for i,(s,e) in enumerate(offs) if e>c0 and s<c1 and e>s]
# ---------------- Klassifikator ---------------------------------------------
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by".split())
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
# ================= TEIL A — PRAEMISSE: gibt es ueberhaupt einen Cluster? ====
# Cell 27A mass +7.8% Kohaerenz der 317 ueber Zufalls-Experten. Der Dekohaerenz-
# Lauf zeigte: die ABSOLUTE Kohaerenz liegt bei 0.52 - also auch bei der
# Kontrollgruppe. Verdacht: die Router-Matrix hat eine globale Vorzugsrichtung,
# und die 317 haben davon nur etwas mehr. Test: globale Richtung aus ALLEN
# Zeilen entfernen, dann erneut vergleichen.
banned={int(l):sorted(v) for l,v in json.load(open(BANNED_JSON)).items()}
GW={}
for name,mod in model.named_modules():
    m=re.fullmatch(r"model\.layers\.(\d+)\.mlp\.gate",name)
    if m and hasattr(mod,"weight"): GW[int(m.group(1))]=mod.weight
assert GW, "keine Router gefunden"
E=GW[sorted(GW)[0]].shape[0]
rng=np.random.default_rng(SEED)
CTRL={l:sorted(rng.choice([e for e in range(E) if e not in banned.get(l,[])],
      size=len(banned.get(l,[])),replace=False).tolist()) for l in GW if banned.get(l)}
pre_b,pre_c,post_b,post_c,glob=[],[],[],[],[]
for l in sorted(GW):
    ids=banned.get(l,[])
    if len(ids)<2: continue
    G=GW[l].detach().float()
    Gn=G/G.norm(dim=1,keepdim=True).clamp_min(1e-9)
    P,mu=strip_global(Gn)
    glob.append(float((Gn@mu).abs().median()))
    pre_b.append(coh(Gn[ids]));   pre_c.append(coh(Gn[CTRL[l]]))
    post_b.append(coh(P[ids]));   post_c.append(coh(P[CTRL[l]]))
nb_pre=sum(1 for a,b in zip(pre_b,pre_c) if a>b)
nb_post=sum(1 for a,b in zip(post_b,post_c) if a>b)
print("TEIL A — PRAEMISSE (Router-Kohaerenz, %d Layer):"%len(pre_b))
print("  Anteil der globalen Richtung je Zeile: median |cos| = %.3f"%float(np.median(glob)))
print("  VOR  Abzug: banned %.3f | Kontrolle %.3f | banned groesser in %d/%d Layern (p=%.4f)"
      %(np.median(pre_b),np.median(pre_c),nb_pre,len(pre_b),sign_p(nb_pre,len(pre_b))))
print("  NACH Abzug: banned %.3f | Kontrolle %.3f | banned groesser in %d/%d Layern (p=%.4f)"
      %(np.median(post_b),np.median(post_c),nb_post,len(post_b),sign_p(nb_post,len(post_b))))
p_post=sign_p(nb_post,len(post_b))
print("  ->","GRUPPENSTRUKTUR BLEIBT (Cluster ueberlebt den Abzug)" if p_post<0.05 and nb_post>len(post_b)/2
      else "KEIN CLUSTER: der Vorsprung war die globale Richtung - Cell-27A-Praemisse faellt")
# ================= TEIL B — GEOMETRIC LENS: Tiefenauflösung ================
# Portierung von _make_attn_hook (geometric-lens): Token t ist ab start_layer
# blind fuer seinen Kontext (nur Selbst-Attention). Sweep ueber die Voll-
# Attention-Layer der Hybrid-Architektur. Readout: unterdrueckte Fremdschrift-
# Masse an den ersten Antwort-Positionen (Druck, wie Cell 17) - fein und billig.
try: model.set_attn_implementation("eager")
except Exception:
    try: model.config._attn_implementation="eager"
    except Exception: pass
FULL=[int(re.fullmatch(r"model\.layers\.(\d+)\.self_attn",n).group(1))
      for n,_ in model.named_modules() if re.fullmatch(r"model\.layers\.(\d+)\.self_attn",n)]
FULL=sorted(FULL)
ATTN={l:dict(model.named_modules())["model.layers.%d.self_attn"%l] for l in FULL}
print("\nTEIL B — GEOMETRIC LENS | Attention: %s | %d Voll-Attention-Layer %s"
      %(getattr(model.config,"_attn_implementation","?"),len(FULL),FULL))
TAB=PROMPTS[[p for p in PROMPT_IDS if p.startswith("643fdf5d")][0]]
EN_TAB=("| Service Name | Storage Limit |\n|---|---|\n"
 "| Google Drive | Google Drive offers 15 GB of free storage for every account. |")
pre=think_prefix(TAB,"")
enc=tokenizer(pre,return_offsets_mapping=True)
IDS=enc["input_ids"]; L=len(IDS)
c0=len(SCAFF)+TAB.index("local name"); c1=c0+len("local name")
DEC=tok_span(enc["offset_mapping"],c0,c1); Q,K=DEC[-1],DEC[0]
CTRL_TOK=max(3,K-20)
print("  Ziel Q=%d %r | Attribut K=%d %r | Kontroll-Token %d %r"
      %(Q,tokenizer.decode([IDS[Q]]),K,tokenizer.decode([IDS[K]]),
        CTRL_TOK,tokenizer.decode([IDS[CTRL_TOK]])))
HK={"tok":None,"start":None}
def mk_pre(lidx):
    def _pre(mod,args,kwargs):
        if HK["tok"] is None or lidx<HK["start"]: return None
        hs=kwargs.get("hidden_states", args[0] if (args and torch.is_tensor(args[0])) else None)
        if hs is None or hs.shape[1]<=1: return None
        am=kwargs.get("attention_mask",None)
        if not (torch.is_tensor(am) and am.dim()==4 and am.shape[-2]==hs.shape[1]): return None
        neg=torch.finfo(am.dtype).min if am.dtype.is_floating_point else False
        kwargs["attention_mask"]=blind_row(am,HK["tok"],neg)
        return (args,kwargs)
    return _pre
hooks=[ATTN[l].register_forward_pre_hook(mk_pre(l),with_kwargs=True) for l in FULL]
# Fremdschrift-Maske
if MASK_NPZ and os.path.exists(MASK_NPZ):
    _z=np.load(MASK_NPZ); M_script=torch.tensor(_z["script"])
    print("  Vokabular-Maske geladen: %d Fremdschrift-Tokens"%int(M_script.sum()))
else:
    raise RuntimeError("vocab_foreign_masks.npz nicht gefunden - erst Cell 17 laufen lassen")
full=pre+EN_TAB
enc2=tokenizer(full,return_offsets_mapping=True)
IDS2=enc2["input_ids"]; A0=next(i for i,(s,e) in enumerate(enc2["offset_mapping"]) if s>=len(pre) and e>s)
ids2=torch.tensor([IDS2],device=model.device)
@torch.no_grad()
def pressure(tok,start,npos=4):
    HK["tok"],HK["start"]=(None,None) if start is None else (tok,start)
    lg=model(input_ids=ids2).logits[0]
    HK["tok"],HK["start"]=None,None
    V=lg.shape[-1]
    m=M_script.to(lg.device)
    if m.shape[0]<V: m=torch.cat([m,torch.zeros(V-m.shape[0],dtype=torch.bool,device=m.device)])
    vals=[]
    for p in range(A0-1,A0-1+npos):
        pr=torch.softmax(lg[p].float(),-1); vals.append(float(pr[m[:V]].sum()))
    return float(np.mean(vals)),vals
base_p,base_v=pressure(None,None)
print("  Baseline-Druck (Fremdschrift-Masse, Mittel ueber 4 Antwort-Positionen): %.4f  %s"
      %(base_p,["%.4f"%v for v in base_v]))
PROF={}
for nm,tok in [("Q=%d"%Q,Q),("K=%d"%K,K),("Ctrl=%d"%CTRL_TOK,CTRL_TOK)]:
    row=[]
    for st in FULL:
        p,_=pressure(tok,st); row.append((st,p))
    PROF[nm]=row
    print("  %-10s Druck je start_layer: %s"%(nm," ".join("L%d:%.4f"%(l,v) for l,v in row)))
for nm,row in PROF.items():
    t=transition(row,base_p)
    print("  %-10s Uebergang (Druck < 50%% der Baseline) ab start_layer: %s"%(nm,t if t is not None else "keiner"))
# ================= TEIL C — Kipp-Rate an ausgewaehlten Startlayern =========
SEL=[None]+[FULL[0],FULL[len(FULL)//3],FULL[2*len(FULL)//3],FULL[-1]]
SW=("takeover","gloss","latin-switch(fr)")
ids1=torch.tensor([IDS],device=model.device)
@torch.no_grad()
def gen_arm(tok,start,n,max_new):
    outs=[]
    for s in range(0,n,CHUNK):
        b=min(CHUNK,n-s)
        HK["tok"],HK["start"]=(None,None) if start is None else (tok,start)
        o=model.generate(ids1.repeat(b,1),do_sample=True,temperature=1.0,top_p=1.0,top_k=0,
                         repetition_penalty=1.0,max_new_tokens=max_new,
                         pad_token_id=tokenizer.eos_token_id)
        HK["tok"],HK["start"]=None,None
        outs+=[tokenizer.decode(x[L:],skip_special_tokens=True) for x in o]
    return outs
print("\nTEIL C — Kipp-Rate (Ziel-Token %d blind ab start_layer, N=%d):"%(Q,N_ANS))
RC={}
try:
    for st in SEL:
        cls=[classify_answer(x) for x in gen_arm(Q,st,N_ANS,MAX_NEW)]
        k=sum(1 for c in cls if c in SW); RC[st]=(k,N_ANS)
        p,lo,hi=wilson(k,N_ANS)
        print("  start_layer=%-5s rate=%5.1f%% [%4.1f,%4.1f]  %s"
              %("none" if st is None else st,100*p,100*lo,100*hi,dict(collections.Counter(cls))))
finally:
    for h in hooks: h.remove()
    HK["tok"],HK["start"]=None,None
kn=RC[None][0]
kills=[st for st in SEL if st is not None and RC[st][0]<kn and twoprop(RC[st][0],N_ANS,kn,N_ANS)<0.05]
print("\nLESART:")
if kills:
    print("  TIEFE LOKALISIERT: Blindheit ab Layer %s toetet noch (Baseline %d/32);"%(kills,kn))
    print("  ab Layer %d wirkt sie nicht mehr - das Ohr liegt UNTERHALB dieser Grenze."%(max(kills)))
else:
    print("  KEIN EINZELNER STARTLAYER TOETET (Baseline %d/32) - der Transport verteilt"%kn)
    print("  sich ueber die Tiefe; nur die Blindheit ab dem ersten Layer wirkt (Cell 29).")
print("  Teil B liefert die feine Aufloesung (Druck), Teil C die grobe Bestaetigung (Kipp).")
GLENS_RESULTS=dict(praemisse=dict(glob=float(np.median(glob)),
                   pre=(float(np.median(pre_b)),float(np.median(pre_c)),nb_pre),
                   post=(float(np.median(post_b)),float(np.median(post_c)),nb_post,p_post)),
                   profile=PROF,base=base_p,switch={str(k):v for k,v in RC.items()})
wc_save_all()
